# Image Segmentation with CLIPSeg

This notebook demonstrates how to use the `CIDAS/clipseg-rd64-refined` model from Hugging Face for zero-shot image segmentation.

In [ ]:
%pip install -q transformers torch matplotlib pillow

In [ ]:
import torch
from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation
from PIL import Image
import matplotlib.pyplot as plt
import os

# Load model and processor
model_id = "CIDAS/clipseg-rd64-refined"
processor = CLIPSegProcessor.from_pretrained(model_id)
model = CLIPSegForImageSegmentation.from_pretrained(model_id)

In [ ]:
image_folder = "imagens"
image_files = [f for f in os.listdir(image_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

print(f"Found {len(image_files)} images in {image_folder}: {image_files}")

## Perform Segmentation

CLIPSeg is zero-shot, meaning we can provide text prompts to segment specific objects.

In [ ]:
def segment_image(image_path, prompts):
    image = Image.open(image_path).convert("RGB")
    
    # Prepare inputs
    inputs = processor(text=prompts, images=[image] * len(prompts), padding="max_length", return_tensors="pt")
    
    # Forward pass
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Get logits
    preds = outputs.logits
    if len(prompts) == 1:
        preds = preds.unsqueeze(0)
        
    # Plot results
    n_prompts = len(prompts)
    fig, ax = plt.subplots(1, n_prompts + 1, figsize=((n_prompts + 1) * 5, 5))
    
    ax[0].imshow(image)
    ax[0].set_title("Original Image")
    ax[0].axis("off")
    
    for i, prompt in enumerate(prompts):
        # Use sigmoid to get probability map
        mask = torch.sigmoid(preds[i])
        ax[i+1].imshow(mask)
        ax[i+1].set_title(f"Mask for: '{prompt}'")
        ax[i+1].axis("off")
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Example usage with the first image
if image_files:
    img_path = os.path.join(image_folder, image_files[1]) # cats_dog.png usually
    prompts = ["a cat", "a dog", "eyes"]
    segment_image(img_path, prompts)

In [ ]:
# Example with kitchen image
if "kitchen.png" in image_files:
    img_path = os.path.join(image_folder, "kitchen.png")
    prompts = ["knife", "bowl", "vegetables"]
    segment_image(img_path, prompts)